# Layer 3 — BTC/ETH Synthetic Scenario Generator (Corrected v1)

**Purpose:** Inject **controlled experimental anomalies** into the existing verified Layer 1 BTCUSDT and ETHUSDT 1-minute datasets, preserve chronological splits, save outputs permanently, and validate them after reload.

**Research limitations:** Synthetic labels indicate *injected anomalies*, **not confirmed real-world market manipulation**. `hl_spread_ratio` is a candle high–low proxy, **not bid–ask spread**. Without historical order-book add/cancel data, this notebook **cannot simulate or label genuine spoofing**. The three controlled scenarios are (1) trade-flow burst, (2) sell-pressure/price-drop with volume surge, (3) breakout-and-reversal. Neither Layer 1 nor Layer 2 is modified.

**Run sequentially in Colab.** No random/fake input fallback, no BTC-as-ETH fallback. Output directory is created **only after** both inputs and generation checks pass. An existing final output directory is never overwritten.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive/research datasets')
L1 = BASE / 'processed_layer1_final_v1' / 'processed_matrices'
OUT = BASE / 'processed_layer3_final_v1'
INPUTS = {
    'BTCUSDT': L1 / 'btcusdt_l1_feature_matrix_90d.parquet',
    'ETHUSDT': L1 / 'ethusdt_l1_feature_matrix_90d.parquet',
}
SEED = 42
EVENTS_PER_SYMBOL = 38
SPLIT_RATIOS = (0.70, 13/90, 14/90)  # 63/13/14-day intended chronology
REQUIRED = {'open','high','low','close','volume','trades_count',
            'log_returns','trade_flow_imbalance','hl_spread_ratio',
            'rush_order_density','volume_acceleration','large_trade_count'}

source = {}
for symbol, path in INPUTS.items():
    assert path.is_file(), f'Missing {symbol} Layer 1 matrix: {path}'
    df = pd.read_parquet(path)
    missing = REQUIRED - set(df.columns)
    assert not missing, f'{symbol}: missing columns {sorted(missing)}; columns={df.columns.tolist()}'
    assert len(df) == 129_600, f'{symbol}: unexpected length {len(df)}'
    assert not df[list(REQUIRED)].isna().any().any(), f'{symbol}: source has NaNs'
    assert np.isfinite(df[list(REQUIRED)].to_numpy(dtype=float)).all(), f'{symbol}: source has infinities'
    assert (df[['open','high','low','close','volume']] > 0).all().all(), f'{symbol}: invalid prices/volume'
    assert (df['trades_count'] >= 0).all(), f'{symbol}: negative trades'
    if 'datetime' in df.columns:
        ts = pd.to_datetime(df['datetime'], utc=True)
        assert ts.is_monotonic_increasing and ts.is_unique, f'{symbol}: invalid timestamp order'
    source[symbol] = df.copy(deep=True)
    print(f'✅ {symbol}: {df.shape}, source loaded; no changes made')
print('Both independent Layer 1 inputs verified.')

## Controlled injections and feature consistency

Event windows do not overlap and remain within their assigned chronological partition. Price perturbations taper to zero at event edges, rather than permanently shifting the market price series. Downstream rolling effects are explicitly flagged as `post_event_context` so they are not mistaken for clean negatives. Recomputed acceleration uses **the previous 15 bars**, with a floor estimated from training history only. Exact Layer 1 preprocessing conventions should be cross-checked before claiming feature-level parity.

In [ ]:
SCENARIOS = ('SYN_01', 'SYN_02', 'SYN_03')
INTENSITIES = ('Low', 'Medium', 'High')
DURATION = {'Low': 10, 'Medium': 15, 'High': 20}


def recompute_features(df, train_end):
    d = df.copy()
    # Conserve base volume and trade-flow sign; derived buy/sell volumes are estimates.
    tfi = d['trade_flow_imbalance'].clip(-1, 1).astype(float)
    d['buy_vol_estimated'] = d['volume'] * (1 + tfi) / 2
    d['sell_vol_estimated'] = d['volume'] * (1 - tfi) / 2
    d['log_returns'] = np.log(d['close'] / d['close'].shift(1)).fillna(0.0)
    d['hl_spread_ratio'] = (d['high'] - d['low']) / d['close']
    d['rush_order_density'] = d['trades_count'] / d['volume'].clip(lower=1e-12)
    # Causal: history excludes current minute. No look-ahead into future bars.
    history = d['volume'].shift(1).rolling(15, min_periods=15)
    avg, std = history.mean(), history.std()
    train_std = std.iloc[15:train_end].dropna()
    assert len(train_std) > 100 and (train_std >= 0).all()
    # Robust train-derived floor, documented in saved generation_config.
    std_floor = max(float(train_std.quantile(0.05)), 1e-8)
    d['volume_acceleration'] = ((d['volume'] - avg) / std.clip(lower=std_floor)).fillna(0.0)
    d['trade_flow_imbalance'] = tfi
    return d, std_floor


def inject_one(d, start, end, scenario, intensity, ret_iqr):
    ix = np.arange(start, end + 1)
    n = len(ix)
    severity = {'Low': 1., 'Medium': 1.5, 'High': 2.0}[intensity]
    # Zero perturbation at first/last bar: price reconnects to observed trajectory.
    hump = np.sin(np.linspace(0, np.pi, n))
    if scenario == 'SYN_01':
        d.loc[ix, 'trades_count'] = np.ceil(d.loc[ix, 'trades_count'].to_numpy() * (1 + 1.4 * severity)).astype(int)
        d.loc[ix, 'volume'] *= (1 + 0.15 * severity)
        d.loc[ix, 'trade_flow_imbalance'] = np.maximum(d.loc[ix, 'trade_flow_imbalance'].to_numpy(), min(.35 * severity, .9))
    elif scenario == 'SYN_02':
        amplitude = max(2 * ret_iqr * severity, 0.0002)
        d.loc[ix, 'close'] *= np.exp(-amplitude * hump)
        d.loc[ix, 'volume'] *= (1 + .8 * severity)
        d.loc[ix, 'trade_flow_imbalance'] = np.minimum(d.loc[ix, 'trade_flow_imbalance'].to_numpy(), -min(.4 * severity, .9))
    elif scenario == 'SYN_03':
        amplitude = max(2 * ret_iqr * severity, 0.0002)
        # Positive breakout followed by negative reversal; zero at both ends.
        phase = np.linspace(0, 2*np.pi, n)
        d.loc[ix, 'close'] *= np.exp(amplitude * np.sin(phase))
        d.loc[ix, 'volume'] *= (1 + .25 * severity)
    else:
        raise ValueError(scenario)
    if scenario != 'SYN_01':
        # Restore open-to-previous-close continuity inside modified event.
        d.loc[ix[1:], 'open'] = d.loc[ix[:-1], 'close'].to_numpy()
        # At first bar retain its observed open; at last bar the close is original.
        d.loc[ix, 'high'] = np.maximum.reduce([d.loc[ix, 'high'].to_numpy(),
                                               d.loc[ix, 'open'].to_numpy(), d.loc[ix, 'close'].to_numpy()])
        d.loc[ix, 'low'] = np.minimum.reduce([d.loc[ix, 'low'].to_numpy(),
                                              d.loc[ix, 'open'].to_numpy(), d.loc[ix, 'close'].to_numpy()])


def generate_for_symbol(symbol, original):
    n = len(original)
    train_end = round(n * SPLIT_RATIOS[0])
    val_end = round(n * sum(SPLIT_RATIOS[:2]))
    boundaries = {'Train': (0, train_end), 'Validation': (train_end, val_end), 'Test': (val_end, n)}
    r = original['log_returns'].iloc[:train_end].dropna().to_numpy(dtype=float)
    ret_iqr = float(np.quantile(r, .75) - np.quantile(r, .25))
    assert np.isfinite(ret_iqr) and ret_iqr > 0
    d = original.copy(deep=True).reset_index(drop=True)
    d['experimental_ground_truth_label'] = np.int8(0)
    d['scenario_type'] = 'Normal_Background'
    d['intensity'] = 'None'
    d['event_id'] = ''
    d['post_event_context'] = np.int8(0)
    d['partition'] = np.where(d.index < train_end, 'Train',
                               np.where(d.index < val_end, 'Validation', 'Test'))
    quotas = {'Train': 24, 'Validation': 7, 'Test': 7}
    rng = np.random.default_rng(SEED + (0 if symbol == 'BTCUSDT' else 1))
    events = []
    event_number = 0
    for split, (left, right) in boundaries.items():
        count = quotas[split]
        # Well-spaced candidate starts, >=60 bars from partition edges.
        candidates = np.arange(left + 65, right - 65, 65)
        assert len(candidates) >= count, f'{symbol}/{split}: insufficient source windows'
        starts = sorted(rng.choice(candidates, size=count, replace=False).tolist())
        for start in starts:
            scenario = SCENARIOS[event_number % len(SCENARIOS)]
            intensity = INTENSITIES[(event_number // len(SCENARIOS)) % len(INTENSITIES)]
            duration = DURATION[intensity]
            end = start + duration - 1
            assert end + 15 < right
            assert (d.loc[start:end, 'experimental_ground_truth_label'] == 0).all()
            inject_one(d, start, end, scenario, intensity, ret_iqr)
            event_number += 1
            eid = f'SYN_{symbol}_{event_number:03d}'
            d.loc[start:end, 'experimental_ground_truth_label'] = 1
            d.loc[start:end, 'scenario_type'] = scenario
            d.loc[start:end, 'intensity'] = intensity
            d.loc[start:end, 'event_id'] = eid
            # Rolling-feature contamination: exclude from clean negatives in training.
            d.loc[end + 1:end + 15, 'post_event_context'] = 1
            events.append({'event_id': eid, 'symbol': symbol, 'partition': split,
                           'scenario_id': scenario, 'intensity': intensity,
                           'start_index': int(start), 'end_index': int(end),
                           'duration_bars': int(duration), 'synthetic_label': 1,
                           'start_datetime': str(original.iloc[start]['datetime']) if 'datetime' in original.columns else None,
                           'end_datetime': str(original.iloc[end]['datetime']) if 'datetime' in original.columns else None})
    assert len(events) == EVENTS_PER_SYMBOL, f'Unexpected count: {len(events)}'
    d, floor = recompute_features(d, train_end)
    return d, events, {'train_end': train_end, 'val_end': val_end,
                        'training_return_iqr': ret_iqr, 'train_derived_acceleration_std_floor': floor}

synthetic = {}
metadata = []
per_symbol_config = {}
for symbol, original in source.items():
    synthetic[symbol], events, cfg = generate_for_symbol(symbol, original)
    metadata.extend(events)
    per_symbol_config[symbol] = cfg
    print(f'✅ {symbol}: {len(events)} events; shape={synthetic[symbol].shape}; split={pd.Series([e["partition"] for e in events]).value_counts().to_dict()}')
assert len(metadata) == 76


## Strict checks before saving

The validation fails loudly on NaN/Inf, OHLC violations, split leakage, mislabeled events, incorrect event count, or any unexpected modifications to the source. It also checks that **only synthetic columns and explicitly engineered columns** differ outside injected event windows.

In [ ]:
def validate(synthetic, source, metadata):
    assert len(metadata) == 76
    ids = [m['event_id'] for m in metadata]
    assert len(ids) == len(set(ids))
    report = {'total_events': len(metadata), 'symbols': {}, 'checks_passed': True}
    for symbol in INPUTS:
        d = synthetic[symbol]
        orig = source[symbol].reset_index(drop=True)
        ev = [m for m in metadata if m['symbol'] == symbol]
        assert len(ev) == 38
        assert len(d) == len(orig)
        nums = d.select_dtypes(include='number')
        assert not d.isna().any().any(), f'{symbol}: NaN'
        assert np.isfinite(nums.to_numpy(dtype=float)).all(), f'{symbol}: Inf'
        assert (d[['open','high','low','close','volume']] > 0).all().all(), f'{symbol}: non-positive OHLCV'
        assert (d['high'] >= d[['open','close','low']].max(axis=1)).all(), f'{symbol}: high boundary'
        assert (d['low'] <= d[['open','close','high']].min(axis=1)).all(), f'{symbol}: low boundary'
        assert d['experimental_ground_truth_label'].isin([0,1]).all()
        assert int(d['experimental_ground_truth_label'].sum()) == sum(e['duration_bars'] for e in ev)
        assert set(d.loc[d['experimental_ground_truth_label'] == 1, 'event_id']) == {e['event_id'] for e in ev}
        for e in ev:
            a,b = e['start_index'],e['end_index']
            assert d.loc[a:b, 'partition'].eq(e['partition']).all(), 'Split boundary crossed'
            assert d.loc[a:b, 'event_id'].eq(e['event_id']).all(), 'Event label mismatch'
        # Original prices, volume, trade counts, imbalance must be unchanged outside injected windows.
        clean = d['experimental_ground_truth_label'].eq(0)
        for col in ['open','high','low','close','volume','trades_count','trade_flow_imbalance']:
            assert np.allclose(d.loc[clean,col], orig.loc[clean,col], rtol=1e-11, atol=1e-10), f'{symbol}: {col} changed outside events'
        report['symbols'][symbol] = {
            'rows': len(d), 'events': len(ev),
            'labeled_anomalous_bars': int(d['experimental_ground_truth_label'].sum()),
            'post_event_context_bars': int(d['post_event_context'].sum()),
            'split_event_counts': {s: sum(e['partition']==s for e in ev) for s in ('Train','Validation','Test')},
            'scenario_counts': {s: sum(e['scenario_id']==s for e in ev) for s in SCENARIOS},
            'missing_values': int(d.isna().sum().sum())}
    return report

validation_report = validate(synthetic, source, metadata)
print(json.dumps(validation_report, indent=2))
print('✅ ALL PRE-SAVE VALIDATIONS PASSED')

## Save to Google Drive (non-destructive)

This cell **refuses to overwrite** an existing final output folder. If a prior partial attempt exists, inspect it manually first. Writes to a temporary sibling folder, reloads every file, and only then renames to the final folder.

In [ ]:
import joblib
import shutil

TMP = BASE / 'processed_layer3_final_v1__staging'
assert not OUT.exists(), f'Final output already exists; refusing overwrite: {OUT}'
assert not TMP.exists(), f'Staging directory exists; inspect it before retrying: {TMP}'
TMP.mkdir(parents=True)
try:
    for symbol, name in [('BTCUSDT', 'synthetic_btcusdt.parquet'),
                         ('ETHUSDT', 'synthetic_ethusdt.parquet')]:
        synthetic[symbol].to_parquet(TMP / name, index=False)
    (TMP / 'event_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    config = {'seed': SEED, 'events_per_symbol': EVENTS_PER_SYMBOL,
              'scenario_descriptions': {
                  'SYN_01': 'Trade-count burst with mild volume surge and buy imbalance',
                  'SYN_02': 'Synthetic transient price drop with volume surge and sell imbalance',
                  'SYN_03': 'Synthetic transient breakout followed by reversal'},
              'limitations': ['Experimental injected anomalies, not verified manipulation',
                              'No spoofing labels without actual order-book cancellations',
                              'High-low spread proxy is not bid-ask spread',
                              'Recomputed volume acceleration may differ from Layer 1 pipeline'],
              'split_ratios': SPLIT_RATIOS, 'input_paths': {k: str(v) for k,v in INPUTS.items()},
              'per_symbol': per_symbol_config}
    (TMP / 'generation_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    (TMP / 'validation_report.json').write_text(json.dumps(validation_report, indent=2), encoding='utf-8')
    # Reopen the persisted files before publishing them.
    for symbol, name in [('BTCUSDT', 'synthetic_btcusdt.parquet'),
                         ('ETHUSDT', 'synthetic_ethusdt.parquet')]:
        reloaded = pd.read_parquet(TMP / name)
        pd.testing.assert_frame_equal(reloaded, synthetic[symbol].reset_index(drop=True), check_dtype=True)
    assert len(json.loads((TMP / 'event_metadata.json').read_text(encoding='utf-8'))) == 76
    assert json.loads((TMP / 'validation_report.json').read_text(encoding='utf-8'))['checks_passed']
    TMP.rename(OUT)
except Exception:
    print(f'⚠️ Save/verification failed; staging retained for inspection: {TMP}')
    raise
print(f'🎉 VERIFIED AND SAVED: {OUT}')
for p in sorted(OUT.iterdir()):
    print(f'  {p.name}: {p.stat().st_size:,} bytes')

## Visual spot-check of actual event locations (optional)

Plot one **real metadata-indexed** event from each scenario, rather than hardcoded example ranges.

In [ ]:
import matplotlib.pyplot as plt
symbol = 'BTCUSDT'
d = synthetic[symbol]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, scenario in zip(axes, SCENARIOS):
    event = next(e for e in metadata if e['symbol']==symbol and e['scenario_id']==scenario)
    a, b = event['start_index'], event['end_index']
    left, right = max(a-20,0), min(b+21,len(d))
    y = 'rush_order_density' if scenario=='SYN_01' else 'close'
    ax.plot(np.arange(left,right), d[y].iloc[left:right].to_numpy(), label=y)
    ax.axvspan(a,b,alpha=.18,label=f'{scenario} injected window')
    ax.set_title(f'{symbol} | {scenario} | {event["intensity"]} | {event["event_id"]}')
    ax.legend()
    ax.grid(True,alpha=.2)
plt.tight_layout()
plt.show()